In [ ]:
# Cell 1: Install dependencies
!pip install -q scanpy anndata igraph leidenalg scikit-learn scipy requests pertpy

In [ ]:
# Cell 2: Mount Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT = '/content/drive/MyDrive/CellJEPA_results/multitissue_universal/multitissue_universal_jepa_final.pt'
VOCAB_FILE  = '/content/drive/MyDrive/CellJEPA_results/universal_vocab/universal_gene_names.json'
RESULTS_DIR = '/content/drive/MyDrive/CellJEPA_results/traj_modes/'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Drive mounted.')
print('Checkpoint:', CHECKPOINT)
print('Vocab file:', VOCAB_FILE)
print('Results dir:', RESULTS_DIR)

In [ ]:
# Cell 3: Upload project files
# Upload all .py files to /content/ before running the cells below.
# Required files:
#   cell_jepa.py, cell_sigreg.py, losses.py, preprocessing.py,
#   trainer.py, metrics.py, perturb_metrics.py, compare_traj_modes.py
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# Cell 4: Smoke test — 3 perturbations, 1 epoch, L_max=100, CPU (~2 min)
# L_max is automatically reduced to 100 in smoke test mode for speed.
# Run without vocab alignment so it works offline / without the checkpoint.
import subprocess

result = subprocess.run(
    ['python3', '-u', '/content/compare_traj_modes.py',
     '--smoke_test',
     '--device', 'cpu',
     '--results_file', 'results_traj_modes_smoke.txt'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-3000:])

In [ ]:
# Cell 4b: Expressed gene diagnostics — CPU, ~2 min
# Reports the distribution of expressed genes per cell after universal vocab alignment.
# Use this to confirm L_max=600 gives adequate coverage before the full run.
# Requires: Cell 4 (smoke test) completed so adamson2016.h5ad is cached locally.
import sys, json
import numpy as np
sys.path.insert(0, '/content')

from compare_traj_modes import load_adamson, align_to_universal_vocab

ctrl_matrix, pert_matrix, pert_ids_cell, conditions, pert_vocab, gene_names, ctrl_label = \
    load_adamson(n_hvg=2000, smoke_test=False)

ctrl_matrix, pert_matrix, vocab_size, gene_vocab = align_to_universal_vocab(
    ctrl_matrix, pert_matrix, gene_names, VOCAB_FILE
)

# Per-cell union of expressed genes (matches PerturbationDataset.__getitem__ logic)
expressed_counts = ((ctrl_matrix > 0) | (pert_matrix > 0)).sum(axis=1)

print(f'Expressed genes per cell (union of ctrl + pert after alignment):')
print(f'  Min:    {expressed_counts.min()}')
print(f'  Median: {int(np.median(expressed_counts))}')
print(f'  Mean:   {expressed_counts.mean():.0f}')
print(f'  p75:    {int(np.percentile(expressed_counts, 75))}')
print(f'  p90:    {int(np.percentile(expressed_counts, 90))}')
print(f'  Max:    {expressed_counts.max()}')
print(f'')
print(f'Universal vocab size: {vocab_size - 2} genes')
print(f'L_max=600 fully covers {(expressed_counts <= 600).mean()*100:.1f}% of cells')
print(f'L_max=400 fully covers {(expressed_counts <= 400).mean()*100:.1f}% of cells')

In [ ]:
# Cell 5: Full run — all 3 modes, pretrained backbone + vocab alignment, A100 (~45-60 min)
# --vocab_file maps Adamson gene names to their universal vocab token IDs so the
# pre-trained gene embeddings are correctly aligned with the checkpoint.
import subprocess, time, threading

t0 = time.time()
proc = subprocess.Popen(
    ['python3', '-u', '/content/compare_traj_modes.py',
     '--pretrain_checkpoint', CHECKPOINT,
     '--vocab_file', VOCAB_FILE,
     '--device', 'cuda',
     '--n_epochs', '15',
     '--batch_size', '64',
     '--results_file', 'results_traj_modes.txt'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True, bufsize=1
)

def stream(pipe):
    for line in pipe:
        print(line, end='', flush=True)

t_out = threading.Thread(target=stream, args=(proc.stdout,))
t_err = threading.Thread(target=stream, args=(proc.stderr,))
t_out.start(); t_err.start()
t_out.join();  t_err.join()

rc = proc.wait()
if rc != 0:
    print(f'\n*** PROCESS EXITED WITH CODE {rc} — see stderr above ***')
else:
    print(f'\nDone in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 5b: Resume — run only specific modes (e.g. after Absolute already completed)
# Edit MODES to the modes you still need to run.
import subprocess, time, threading

MODES = 'delta,traj'   # <-- edit this

t0 = time.time()
proc = subprocess.Popen(
    ['python3', '-u', '/content/compare_traj_modes.py',
     '--pretrain_checkpoint', CHECKPOINT,
     '--vocab_file', VOCAB_FILE,
     '--device', 'cuda',
     '--n_epochs', '15',
     '--batch_size', '64',
     '--modes', MODES,
     '--results_file', 'results_traj_modes_resume.txt'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True, bufsize=1
)

def stream(pipe):
    for line in pipe:
        print(line, end='', flush=True)

t_out = threading.Thread(target=stream, args=(proc.stdout,))
t_err = threading.Thread(target=stream, args=(proc.stderr,))
t_out.start(); t_err.start()
t_out.join();  t_err.join()

rc = proc.wait()
if rc != 0:
    print(f'\n*** PROCESS EXITED WITH CODE {rc} ***')
else:
    print(f'\nDone in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 6: Display results table and bar chart
import os, re
import numpy as np
import matplotlib.pyplot as plt

for fname in ['results_traj_modes_smoke.txt', 'results_traj_modes.txt', 'results_traj_modes_resume.txt']:
    if os.path.exists(fname):
        print(f'--- {fname} ---')
        print(open(fname).read())

def parse_results(path):
    if not os.path.exists(path):
        return {}
    data = {}
    for line in open(path):
        nums = re.findall(r'-?\d+\.\d{4,}', line)
        if len(nums) >= 4:
            label = line[:28].strip()
            if label and not label.startswith(('-', '=', 'M', 'P')):
                data[label] = {
                    'pearson':       float(nums[0]),
                    'pearson_delta': float(nums[1]),
                    'top20':         float(nums[2]),
                    'mse':           float(nums[3]),
                }
    return data

data = parse_results('results_traj_modes.txt')
data.update(parse_results('results_traj_modes_resume.txt'))
if not data:
    data = parse_results('results_traj_modes_smoke.txt')

if data:
    modes      = list(data.keys())
    metrics    = ['pearson', 'pearson_delta', 'top20']
    met_labels = ['Pearson r', 'Pearson Δ', 'Top-20 DEG Δ']
    colors     = ['#4C72B0', '#DD8452', '#55A868']

    x  = np.arange(len(modes))
    bw = 0.22
    fig, ax = plt.subplots(figsize=(9, 5))

    for mi, (met, mlabel, col) in enumerate(zip(metrics, met_labels, colors)):
        vals   = [data[m][met] for m in modes]
        offset = (mi - 1) * bw
        bars   = ax.bar(x + offset, vals, bw, label=mlabel, color=col, alpha=0.85)
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bw / 2, h + 0.005, f'{h:.3f}',
                    ha='center', va='bottom', fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(modes, fontsize=11)
    ax.set_title('CellJEPA Perturbation Mode Comparison — Adamson 2016', fontsize=12)
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)
    ax.set_axisbelow(True)

    plt.tight_layout()
    plt.savefig('traj_modes_comparison.png', dpi=150)
    plt.show()
    print('Saved traj_modes_comparison.png')
else:
    print('No results to plot — did Cell 5 complete?')

In [ ]:
# Cell 7: Save results to Drive
import shutil

for f in ['results_traj_modes.txt', 'results_traj_modes_resume.txt',
          'results_traj_modes_smoke.txt', 'traj_modes_comparison.png']:
    if os.path.exists(f):
        shutil.copy(f, RESULTS_DIR)
        print(f'Copied {f}')
    else:
        print(f'Not found: {f} (skipping)')

print(f'Done. Files at {RESULTS_DIR}')